In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


# =========================
# CONFIG
# =========================

CSV_PATH = "raw_data.csv"

HORIZON = 60
LATENT_DIM = 12
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-3

BETA_KL = 1e-3
LAMBDA_SMOOTH = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# DATASET
# =========================

class YieldChangeDataset(Dataset):
    def __init__(self, dy_norm, horizon=60):
        self.x = []

        for i in range(len(dy_norm) - horizon + 1):
            window = dy_norm[i:i+horizon]
            self.x.append(window)

        self.x = torch.tensor(np.array(self.x), dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]


# =========================
# MODEL
# =========================

class MLPVAE(nn.Module):
    def __init__(self, horizon=60, n_maturities=11, latent_dim=12):
        super().__init__()

        self.horizon = horizon
        self.n_maturities = n_maturities
        self.input_dim = horizon * n_maturities

        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, self.input_dim)
        )

    def encode(self, x):
        x_flat = x.reshape(x.size(0), -1)
        h = self.encoder(x_flat)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        out = self.decoder(z)
        return out.reshape(-1, self.horizon, self.n_maturities)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


# =========================
# LOSS
# =========================

def vae_loss(recon, x, mu, logvar):
    recon_loss = F.mse_loss(recon, x)

    kl_loss = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    smooth_loss = F.mse_loss(
        recon[:, 1:, :] - recon[:, :-1, :],
        x[:, 1:, :] - x[:, :-1, :]
    )

    total = recon_loss + BETA_KL * kl_loss + LAMBDA_SMOOTH * smooth_loss

    return total, recon_loss, kl_loss, smooth_loss


# =========================
# LOAD + PREPROCESS DATA
# =========================

df = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df = df.sort_index()

macro_cols = ["GDP", "CPI", "FedRate", "FedFunds"]
df = df.drop(columns=[c for c in macro_cols if c in df.columns], errors="ignore")

df = df.resample("ME").last().dropna()

yield_cols = [c for c in df.columns if "Y_DGS" in c]

if len(yield_cols) != 11:
    print("Warning: expected 11 yield columns, found:", len(yield_cols))

y = df[yield_cols].values.astype(np.float32)

dy = y[1:] - y[:-1]

n = len(dy)
train_end = int(0.75 * n)
val_end = int(0.90 * n)

dy_train = dy[:train_end]
dy_val = dy[train_end:val_end]
dy_test = dy[val_end:]

mu_dy = dy_train.mean(axis=0)
std_dy = dy_train.std(axis=0) + 1e-8

dy_train_norm = (dy_train - mu_dy) / std_dy
dy_val_norm = (dy_val - mu_dy) / std_dy
dy_test_norm = (dy_test - mu_dy) / std_dy

train_ds = YieldChangeDataset(dy_train_norm, HORIZON)
val_ds = YieldChangeDataset(dy_val_norm, HORIZON) if len(dy_val_norm) >= HORIZON else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE) if val_ds else None

print("Monthly rows:", len(df))
print("Train windows:", len(train_ds))
print("Val windows:", len(val_ds) if val_ds else 0)


# =========================
# TRAIN
# =========================

model = MLPVAE(
    horizon=HORIZON,
    n_maturities=len(yield_cols),
    latent_dim=LATENT_DIM
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()

    train_losses = []

    for x in train_loader:
        x = x.to(DEVICE)

        recon, mu, logvar = model(x)
        loss, rec, kl, smooth = vae_loss(recon, x, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    if val_loader:
        with torch.no_grad():
            for x in val_loader:
                x = x.to(DEVICE)
                recon, mu, logvar = model(x)
                loss, _, _, _ = vae_loss(recon, x, mu, logvar)
                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "simple_vae_yield_changes.pt")

    else:
        val_loss = np.nan
        torch.save(model.state_dict(), "simple_vae_yield_changes.pt")

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train loss: {np.mean(train_losses):.5f} | "
        f"Val loss: {val_loss:.5f}"
    )


# =========================
# GENERATE SCENARIOS
# =========================

@torch.no_grad()
def generate_scenarios(model, start_curve, n_scenarios=100):
    model.eval()

    z = torch.randn(n_scenarios, LATENT_DIM).to(DEVICE)

    dy_norm_generated = model.decode(z).cpu().numpy()

    dy_generated = dy_norm_generated * std_dy.reshape(1, 1, -1) + mu_dy.reshape(1, 1, -1)

    levels = start_curve.reshape(1, 1, -1) + np.cumsum(dy_generated, axis=1)

    return levels


start_curve = y[-1]

scenarios = generate_scenarios(
    model=model,
    start_curve=start_curve,
    n_scenarios=200
)

print("Generated scenarios shape:", scenarios.shape)


# =========================
# SAVE OUTPUT
# =========================

rows = []

start_date = df.index[-1]
future_dates = pd.date_range(
    start=start_date + pd.offsets.MonthEnd(1),
    periods=HORIZON,
    freq="ME"
)

for s in range(scenarios.shape[0]):
    for t in range(HORIZON):
        row = {
            "Scenario_ID": s + 1,
            "Month": t + 1,
            "Date": future_dates[t]
        }

        for j, col in enumerate(yield_cols):
            row[col] = scenarios[s, t, j]

        rows.append(row)

scenario_df = pd.DataFrame(rows)
scenario_df.to_csv("simple_vae_yield_scenarios.csv", index=False)

print("Saved: simple_vae_yield_scenarios.csv")


# =========================
# QUICK PLOT
# =========================

maturity_to_plot = 0

plt.figure(figsize=(10, 5))

for s in range(20):
    plt.plot(
        range(1, HORIZON + 1),
        scenarios[s, :, maturity_to_plot],
        alpha=0.4
    )

plt.title(f"Generated scenarios for {yield_cols[maturity_to_plot]}")
plt.xlabel("Month")
plt.ylabel("Yield")
plt.show()